In [ ]:
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as pyp

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso

from sklearn.metrics import root_mean_squared_error
import pickle

import mlflow

import xgboost as xgb
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope


In [2]:
mlflow.set_tracking_uri("sqlite:////workspaces/mlops-zoomcamp/mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='/workspaces/mlops-zoomcamp/03/mlruns/1', creation_time=1790963508959, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1790963508959, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>

In [4]:
categorical = ['PU_DO']
numerical = ['trip_distance']

def read_dataframe(filename):

    df = pd.read_parquet(filename)

    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds()/60)

    df = df[((df.duration >= 1) & (df.duration <= 60))]
    
    df['PU_DO'] = df['PULocationID'].astype(str) + '_' + df['DOLocationID'].astype(str)
    
    return df

In [5]:
df_train = read_dataframe('./data/green_tripdata_2026-01.parquet')
df_val = read_dataframe('./data/green_tripdata_2026-02.parquet')

In [6]:
dv = DictVectorizer()
train_dicts = df_train[categorical+numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

# updating indicies to be able to run Lasso regression (fit needs 32 bit ints)
X_train.indices = X_train.indices.astype('int32')
X_train.indptr = X_train.indptr.astype('int32')
X_val.indices = X_val.indices.astype('int32')
X_val.indptr = X_val.indptr.astype('int32')

target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)
root_mean_squared_error(y_val, y_pred)

7.10876663907057

In [15]:
with mlflow.start_run():
    mlflow.set_tag("developer", "Gurpal")
    mlflow.log_param("train-data-path", "./data/green_tripdata_2026-01.parquet")
    mlflow.log_param("valid-data-path", "./data/green_tripdata_2026-02.parquet")

    alpha = 0.1
    mlflow.log_param("alpha", alpha)

    lr = Lasso(alpha)
    lr.fit(X_train, y_train)
    y_pred = lr.predict(X_val)

    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    mlflow.log_artifact(local_path="models/lin_reg.bin", artifact_path="models_pickle")
    
    mlflow.end_run()

2026/10/02 20:43:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [18]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

mlflow.xgboost.autolog(disable=True)

In [9]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

In [ ]:
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:squarederror',   # fixes the deprecation warning too
    'tree_method': 'hist',             # faster and lighter on memory
    'seed': 42
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials()
)

[0]	validation-rmse:6.27132                           
[1]	validation-rmse:6.19153                           
[2]	validation-rmse:6.14125                           
[3]	validation-rmse:6.14411                           
[4]	validation-rmse:6.13879                           
[5]	validation-rmse:6.14363                           
[6]	validation-rmse:6.14250                           
[7]	validation-rmse:6.14620                           
[8]	validation-rmse:6.14455                           
[9]	validation-rmse:6.14454                           
[10]	validation-rmse:6.14652                          
[11]	validation-rmse:6.14686                          
[12]	validation-rmse:6.14968                          
[13]	validation-rmse:6.14476                          
[14]	validation-rmse:6.14465                          
[15]	validation-rmse:6.14622                          
[16]	validation-rmse:6.14691                          
[17]	validation-rmse:6.14703                          
[18]	valid

In [19]:
best_params = {
    'learning_rate': 0.35585138837330477,
    'max_depth': 7,
    'min_child_weight': 1.6326562179291448,
    'objective': 'reg:squarederror',
    'reg_alpha': 0.10108911319056418,
    'reg_lambda': 0.020199,
    'seed': 42,
    'tree_method': 'hist'
}

mlflow.log_params(best_params)

booster = xgb.train(
    params = best_params,
    dtrain = train,
    num_boost_round = 1000,
    evals = [(valid, "validation")],
    early_stopping_rounds=50
)

y_pred = booster.predict(valid)
rmse = root_mean_squared_error(y_val, y_pred)
mlflow.log_metric("rmse", rmse)

with open("models/preprocessor.b", "wb") as f_out:
    pickle.dump(dv, f_out)

mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

[0]	validation-rmse:8.25396


[1]	validation-rmse:7.17032
[2]	validation-rmse:6.64797
[3]	validation-rmse:6.40526
[4]	validation-rmse:6.28749
[5]	validation-rmse:6.23509
[6]	validation-rmse:6.20404
[7]	validation-rmse:6.18515
[8]	validation-rmse:6.17152
[9]	validation-rmse:6.16317
[10]	validation-rmse:6.15990
[11]	validation-rmse:6.15895
[12]	validation-rmse:6.15380
[13]	validation-rmse:6.14946
[14]	validation-rmse:6.14525
[15]	validation-rmse:6.14048
[16]	validation-rmse:6.13678
[17]	validation-rmse:6.13360
[18]	validation-rmse:6.13186
[19]	validation-rmse:6.12912
[20]	validation-rmse:6.12616
[21]	validation-rmse:6.12527
[22]	validation-rmse:6.12533
[23]	validation-rmse:6.12211
[24]	validation-rmse:6.12020
[25]	validation-rmse:6.11658
[26]	validation-rmse:6.11261
[27]	validation-rmse:6.10927
[28]	validation-rmse:6.10622
[29]	validation-rmse:6.10627
[30]	validation-rmse:6.10537
[31]	validation-rmse:6.10299
[32]	validation-rmse:6.10152
[33]	validation-rmse:6.09695
[34]	validation-rmse:6.09529
[35]	validation-rmse:6.

2026/10/02 20:59:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [24]:
loaded_model = mlflow.pyfunc.load_model(
    'runs:/ec01fff616a6484bb1f2d5cc5014c2b8/models_mlflow'
)

xgboost_model = mlflow.xgboost.load_model(
    'runs:/ec01fff616a6484bb1f2d5cc5014c2b8/models_mlflow'
)

In [26]:
pred = xgboost_model.predict(valid)

pred[:10]

array([ 5.038775,  8.287124, 25.343729, 13.951615, 16.185984,  8.269009,
       28.312263, 22.02137 , 13.683667,  3.750457], dtype=float32)

In [28]:
mlflow.end_run()

In [29]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import LinearSVR

mlflow.sklearn.autolog()

for model_class in (RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, LinearSVR):

    with mlflow.start_run():

        mlflow.log_param("train-data-path", "./data/green_tripdata_2021-01.csv")
        mlflow.log_param("valid-data-path", "./data/green_tripdata_2021-02.csv")
        mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

        mlmodel = model_class()
        mlmodel.fit(X_train, y_train)

        y_pred = mlmodel.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)

2026/10/02 21:18:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/02 21:18:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/02 21:24:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mec

In [ ]:
with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [ ]:
sb.distplot(y_pred, label='prediction')
sb.distplot(y_val, label='actual')